# 🛡️ FAILSAFE — Phase 3: Data Preprocessing & Pipeline

> **Mentor Note:** Preprocessing is where most real-world ML projects fail silently.
> A model trained on leaky or improperly scaled data will look great on paper and fail
> catastrophically in production. This notebook teaches you to preprocess *correctly*
> — not just get the code running.

---

## What We Are Doing in This Notebook

| Section | Goal |
|---------|------|
| 1. Setup & Load | Reproduce the exact data state from Phase 2 |
| 2. Feature Selection | Decide exactly which columns go into the model |
| 3. Train/Test Split | Why we split BEFORE preprocessing |
| 4. Missing Value Imputation | Strategy selection + why median beats mean |
| 5. Encoding Categoricals | OrdinalEncoder vs OneHotEncoder — the tradeoff |
| 6. Feature Scaling | StandardScaler — when it matters and when it doesn't |
| 7. sklearn Pipeline | Chaining steps, preventing leakage, saving artifacts |
| 8. Pipeline Validation | Verify the output is correct before modeling |
| 9. Feature Engineering | One targeted engineered feature (G2-G1 delta) |
| 10. Save the Preprocessor | joblib serialisation for production serving |
| 11. Interview Q&A | Questions generated directly from this notebook |

---

## Why Preprocessing Matters

Raw data cannot go into a model. Every ML algorithm expects:
- **Numbers only** — no strings, no `yes/no`, no `GP`/`MS`
- **No NaN values** — most algorithms will crash or silently produce `nan` predictions
- **Consistent scale** — gradient-based models (Logistic Regression) suffer badly from unscaled features

And critically:
- **No data leakage** — preprocessing statistics (mean, std, encoder categories)
  must be computed on TRAINING data only, then applied to test data.
  Violating this = your test accuracy is a lie.


---
## Section 1 — Setup & Load Data

We start from the raw CSV, reproduce the exact same target variable as Phase 2,
and confirm our starting state before touching any preprocessing.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
os.makedirs('../plots', exist_ok=True)
os.makedirs('../models', exist_ok=True)

ACCENT      = '#4f8ef7'
RISK_COLORS = {0: '#10b981', 1: '#ef4444'}

# ── Load raw data ──────────────────────────────────────────────────────────────
df = pd.read_csv('../data/student-mat.csv', sep=';')

# ── Create target (identical to Phase 2 — must be reproducible) ───────────────
df['at_risk'] = (df['G3'] < 10).astype(int)

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'at_risk=1 (fail): {df["at_risk"].sum()} ({df["at_risk"].mean()*100:.1f}%)')
print(f'at_risk=0 (pass): {(df["at_risk"]==0).sum()} ({(df["at_risk"]==0).mean()*100:.1f}%)')
print('\n✅ Data loaded. Ready for preprocessing.')

---
## Section 2 — Feature Selection

### What goes in, what stays out

Before we preprocess anything, we must explicitly decide which features enter the model.
This is a **domain + leakage** decision, not a statistical one.

**Rules for inclusion:**
1. Feature must be available **before** the final exam (no G3)
2. Feature must carry some predictive signal (from Phase 2 EDA)
3. Feature must not be a direct proxy of the target (no leakage)

**What we DROP and why:**

| Column | Reason Dropped |
|--------|---------------|
| `G3` | **TARGET LEAKAGE** — this IS the label. Never include. |

**What we KEEP even though EDA showed weak signal:**
- `health`, `famrel`, `freetime` — weak alone, may contribute in combination
- `nursery`, `Pstatus` — low signal but no leakage risk, cost is near zero
- Better to let the model decide via SHAP than to manually remove weak features

**Interview Q: 'Why did you keep all features?'**  
**Answer:** 'Feature selection by hand can introduce bias. With 32 features and 395 rows,
we're not at risk of the curse of dimensionality. I let XGBoost's regularisation
(lambda, alpha) handle irrelevant features automatically. SHAP values confirmed post-training
which features actually mattered.'

In [ ]:
# ── Feature lists ─────────────────────────────────────────────────────────────
# Splitting by data type is REQUIRED for our ColumnTransformer:
# numeric → impute with median → scale
# categorical → impute with mode → ordinal encode

NUMERIC_FEATURES = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
    'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences',
    'G1', 'G2'
]

CATEGORICAL_FEATURES = [
    'school', 'sex', 'address', 'famsize', 'Pstatus',
    'Mjob', 'Fjob', 'reason', 'guardian',
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# X = feature matrix, y = target vector
X = df[ALL_FEATURES].copy()
y = df['at_risk'].copy()

print('FEATURE SELECTION SUMMARY')
print('=' * 50)
print(f'Numeric features    : {len(NUMERIC_FEATURES)}')
print(f'Categorical features: {len(CATEGORICAL_FEATURES)}')
print(f'Total features      : {len(ALL_FEATURES)}')
print(f'Dropped             : G3 (target leakage)')
print(f'X shape: {X.shape} | y shape: {y.shape}')
print()

# Sanity check: confirm X has NO target column
assert 'G3' not in X.columns, '⚠️  DATA LEAKAGE: G3 found in X!'
assert 'at_risk' not in X.columns, '⚠️  DATA LEAKAGE: at_risk found in X!'
print('✅ Leakage check passed: G3 and at_risk are NOT in X.')

---
## Section 3 — Train/Test Split

### The Golden Rule: Split BEFORE Preprocessing

This is the most common mistake made by ML beginners — and it's asked in almost
every DS/ML interview.

```
❌ WRONG ORDER (data leakage):
   1. Scale all X with scaler.fit_transform(X)   ← scaler sees test set statistics!
   2. Split into X_train, X_test
   3. Train model on X_train
   → Your test accuracy is optimistically biased — the model has indirectly
     seen the test set through the scaler's mean/std.

✅ CORRECT ORDER (no leakage):
   1. Split X, y into X_train, X_test          ← split first
   2. preprocessor.fit(X_train)                ← learn stats from train only
   3. X_train_t = preprocessor.transform(X_train)
   4. X_test_t  = preprocessor.transform(X_test)  ← apply train stats to test
   5. Train model on X_train_t
```

### stratify=y — Why It's Mandatory for Imbalanced Data

Without stratify, a random split might put most at-risk students in the training
set and almost none in the test set — making your test recall look artificially high.

`stratify=y` guarantees the same ~33% at-risk ratio in both train and test splits.

In [ ]:
# ── Train/Test Split ──────────────────────────────────────────────────────────
# test_size=0.2  → 80/20 split = 316 train, 79 test
# random_state=42 → reproducibility (same split every run)
# stratify=y      → preserves class ratio in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('TRAIN/TEST SPLIT RESULTS')
print('=' * 50)
print(f'Training set : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print(f'Train at_risk rate: {y_train.mean():.4f} ({y_train.mean()*100:.1f}%)')
print(f'Test  at_risk rate: {y_test.mean():.4f}  ({y_test.mean()*100:.1f}%)')
print(f'Original   rate   : {y.mean():.4f}  ({y.mean()*100:.1f}%)')
print()

# Verify stratification worked — rates should be nearly identical
rate_diff = abs(y_train.mean() - y_test.mean())
if rate_diff < 0.02:
    print(f'✅ Stratification verified: difference = {rate_diff:.4f} (< 2%)')
else:
    print(f'⚠️  Unexpected class rate difference: {rate_diff:.4f}')

In [ ]:
# ── Visualise the split ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Train/Test Split — Class Distribution Check', fontweight='bold', fontsize=12)

for ax, (label, data) in zip(axes, [('Full Dataset', y), ('Training Set', y_train), ('Test Set', y_test)]):
    counts = data.value_counts().sort_index()
    bars = ax.bar(['Safe', 'At Risk'], counts.values,
                  color=[RISK_COLORS[0], RISK_COLORS[1]],
                  edgecolor='#0f1117', width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontweight='bold', fontsize=10)
    ax.set_title(f'{label}\n(n={len(data)}, risk={data.mean()*100:.1f}%)', fontweight='bold', fontsize=9)
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts.values) * 1.2)

plt.tight_layout()
plt.savefig('../plots/14_train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 All three bars show the same ~67/33 ratio — stratification worked perfectly.')

---
## Section 4 — Missing Value Imputation

### Why impute even though UCI data has no missing values?

Two reasons:
1. **Production safety:** When faculty upload their own CSVs, they WILL have missing
   values — students who didn't submit, absent on test day, etc.
   Our pipeline must handle this without crashing.
2. **Interview signal:** Building a robust pipeline even when not strictly required
   shows production-thinking, not just notebook-thinking.

### Median vs Mean — Why It Matters

```
absences distribution: [0, 0, 0, 2, 4, 6, 8, 12, 14, 75]

mean   = (0+0+0+2+4+6+8+12+14+75) / 10 = 12.1   ← pulled up by outlier 75
median = middle value = 5.0                       ← robust to outliers

If a student is missing absences and we impute 12.1,
the model thinks they have high absences → might flag them as at-risk incorrectly.
Median (5.0) is a much safer estimate of the 'typical' student.
```

**Rule of thumb:**
- Symmetric distributions → mean is fine
- Skewed distributions → use median
- Categorical → use mode (most frequent value)

In [ ]:
# ── Demonstrate mean vs median on the absences column ────────────────────────
# This cell is for understanding — actual imputation happens inside the Pipeline

print('MEAN vs MEDIAN IMPUTATION — absences column')
print('=' * 50)
print(f'  Mean   (absences): {X_train["absences"].mean():.2f}')
print(f'  Median (absences): {X_train["absences"].median():.2f}')
print(f'  Skewness         : {X_train["absences"].skew():.2f}')
print(f'  Max value        : {X_train["absences"].max()}')
print()
print('Because absences is heavily right-skewed (skewness > 1),')
print('median is the more robust imputation strategy.')
print()

# Show which numeric features are most skewed (most affected by mean vs median)
skewness = X_train[NUMERIC_FEATURES].skew().abs().sort_values(ascending=False)
print('Top 5 most skewed numeric features (abs skewness):')
for feat, sk in skewness.head(5).items():
    strategy = 'MEDIAN ✅' if sk > 0.5 else 'mean is OK'
    print(f'  {feat:<12}: skew={sk:.2f}  → use {strategy}')

In [ ]:
# ── Simulate imputation to show what would happen ────────────────────────────
# Artificially introduce NaN values in a copy, then show imputation in action

import numpy as np

X_sim = X_train.copy()
# Set 10% of 'absences' to NaN randomly
rng = np.random.default_rng(42)
nan_idx = rng.choice(len(X_sim), size=int(len(X_sim)*0.1), replace=False)
X_sim.loc[X_sim.index[nan_idx], 'absences'] = np.nan

print(f'Simulated NaN count in absences: {X_sim["absences"].isna().sum()}')

# Apply median imputer
from sklearn.impute import SimpleImputer
sim_imputer = SimpleImputer(strategy='median')
sim_imputer.fit(X_sim[['absences']])
filled = sim_imputer.transform(X_sim[['absences']])

print(f'After imputation — NaN count : {np.isnan(filled).sum()}')
print(f'Imputed value used (median)  : {sim_imputer.statistics_[0]:.1f}')
print()
print('✅ All NaN values replaced with the training median.')
print('   The test set will use the SAME median (learned from training only).')

---
## Section 5 — Encoding Categorical Features

ML models require numbers. Categorical strings like `'yes'`, `'no'`, `'GP'`, `'teacher'`
must be converted to integers.

### Two main strategies:

#### OrdinalEncoder (what we use)
Converts each category to an integer: `'no'→0`, `'yes'→1`, `'GP'→0`, `'MS'→1`

```
Before: ['yes', 'no', 'yes', 'yes', 'no']
After:  [  1,    0,    1,    1,    0  ]
```

**Pros:** One column per feature, compact, XGBoost handles it well  
**Cons:** Implies an ordering that doesn't exist (is 'teacher' > 'at_home'?)

#### OneHotEncoder (alternative)
Creates a binary column for each category value.

```
Before: ['GP', 'MS', 'GP']
After:  [[1,0], [0,1], [1,0]]   ← 2 new binary columns
```

**Pros:** No false ordering implied, ideal for Logistic Regression  
**Cons:** Many sparse columns (Mjob alone → 5 columns), memory cost grows

### Our Decision: OrdinalEncoder

**Primary model is XGBoost.** Tree-based models find optimal splits at each node —
they don't assume any ordering. So `'teacher'→4` vs `'at_home'→0` doesn't matter;
XGBoost will split `Mjob < 2` or `Mjob ≥ 3` regardless of assignment order.

**Interview answer:** *'I used OrdinalEncoder because XGBoost's tree splits don't
assume ordinal relationships. For the Logistic Regression baseline I would switch to
OneHotEncoder since LR is sensitive to implied orderings.'*

### handle_unknown='use_encoded_value', unknown_value=-1

This is a critical production safety setting. If a faculty CSV contains a new school
name we've never seen during training, the encoder won't crash — it maps the unknown
value to `-1` so inference can continue.

In [ ]:
# ── Demonstrate OrdinalEncoder vs OneHotEncoder ───────────────────────────────

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import pandas as pd

sample = pd.DataFrame({'higher': ['yes','no','yes','yes','no'],
                       'Mjob':   ['teacher','at_home','health','services','other']})

print('Original sample:')
print(sample.to_string())
print()

# OrdinalEncoder
ord_enc = OrdinalEncoder()
ordinal_result = pd.DataFrame(
    ord_enc.fit_transform(sample),
    columns=sample.columns
)
print('After OrdinalEncoder (one integer per category):')
print(ordinal_result.to_string())
print(f'Shape: {ordinal_result.shape}  ← same 2 columns')
print(f'Mjob categories → {list(enumerate(ord_enc.categories_[1]))}')
print()

# OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
ohe_result = pd.DataFrame(
    ohe.fit_transform(sample),
    columns=ohe.get_feature_names_out()
)
print('After OneHotEncoder (binary column per category):')
print(ohe_result.to_string())
print(f'Shape: {ohe_result.shape}  ← expanded to {ohe_result.shape[1]} columns')

In [ ]:
# ── Show all unique values in each categorical feature ────────────────────────
# Important for understanding what the encoder will learn during fit()

print('CATEGORICAL FEATURE VOCABULARY')
print('=' * 55)
for col in CATEGORICAL_FEATURES:
    unique_vals = sorted(X_train[col].unique().tolist())
    print(f'  {col:<12}: {unique_vals}')

---
## Section 6 — Feature Scaling

### When scaling matters — and when it doesn't

**Does XGBoost need scaling?** NO.  
Tree-based models make splits based on thresholds. Whether `absences` is 14 or
normalised to 1.3 makes no difference — the split at `absences > 10` is the same.

**Does Logistic Regression need scaling?** YES, critically.  
LR uses gradient descent. Without scaling, features with large ranges (absences: 0–93)
dominate the loss function over features with small ranges (traveltime: 1–4).
Gradient updates become wildly unbalanced, and the model converges poorly or not at all.

**Why include scaling anyway if XGBoost doesn't need it?**
1. We train Logistic Regression as a baseline — it needs scaling
2. The same preprocessor.pkl must work for all models consistently
3. Scaling never hurts tree models; it only helps linear models
4. Normalised features make SHAP values slightly more comparable across features

### StandardScaler — Z-score Normalisation

```
z = (x - mean) / std

absences before: [0, 2, 4, 6, 14, 75]
                  mean=16.8, std=18.2
absences after:  [-0.92, -0.81, -0.70, -0.59, -0.15, 3.20]

Result: mean=0, std=1. All features are now on the same scale.
```

**Key:** `scaler.fit(X_train)` computes mean+std FROM TRAINING DATA ONLY.  
`scaler.transform(X_test)` applies those SAME training statistics to the test set.  
This is enforced automatically by the Pipeline.

In [ ]:
# ── Visualise StandardScaler effect on absences ───────────────────────────────

from sklearn.preprocessing import StandardScaler

scaler_demo = StandardScaler()
absences_raw    = X_train[['absences']].values
absences_scaled = scaler_demo.fit_transform(absences_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('StandardScaler Effect on absences', fontweight='bold', fontsize=12)

ax = axes[0]
ax.hist(absences_raw, bins=20, color=ACCENT, edgecolor='#0f1117', alpha=0.85)
ax.set_title(f'Before Scaling\nmean={absences_raw.mean():.1f}, std={absences_raw.std():.1f}',
             fontweight='bold')
ax.set_xlabel('Raw absences value')

ax = axes[1]
ax.hist(absences_scaled, bins=20, color='#a855f7', edgecolor='#0f1117', alpha=0.85)
ax.set_title(f'After Scaling\nmean={absences_scaled.mean():.2f}, std={absences_scaled.std():.2f}',
             fontweight='bold')
ax.set_xlabel('Z-score (standardised)')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6, label='mean=0')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../plots/15_scaling_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Shape is preserved — only the axis changes.')
print('   The right-skew outlier (75 absences) is now 3.2 standard deviations above mean.')
print('   This extreme value will still affect Linear models but not tree-based ones.')

In [ ]:
# ── Feature range comparison: before and after scaling ───────────────────────

scaler_all = StandardScaler()
X_train_num_scaled = scaler_all.fit_transform(X_train[NUMERIC_FEATURES])

before = X_train[NUMERIC_FEATURES].describe().loc[['mean','std','min','max']]
after  = pd.DataFrame(X_train_num_scaled, columns=NUMERIC_FEATURES)\
           .describe().loc[['mean','std','min','max']]

print('BEFORE SCALING (raw ranges):')
print(before.round(2).to_string())
print()
print('AFTER SCALING (all centred at 0, std=1):')
print(after.round(2).to_string())
print()
print('📌 Note: absences had range [0, 75]. After scaling: [-0.92, 3.20].')
print('   Now all numeric features are on the same ~[-3, 3] scale.')

---
## Section 7 — Building the sklearn Pipeline

### What is a Pipeline?

A `Pipeline` is a single object that chains multiple preprocessing steps and optionally
a final estimator. It behaves like a single transformer/model:
- Call `.fit(X_train)` → fits all steps in sequence on training data
- Call `.transform(X)` → applies all fitted steps in sequence
- Call `.fit_transform(X_train)` → both in one call

### ColumnTransformer — applying different pipelines to different column subsets

```
                    ┌────────────────────────────────┐
X (32 columns)  ──► │     ColumnTransformer           │
                    │                                │
  numeric cols ─────►  impute(median) → scale(z)    │
  categorical cols ─►  impute(mode)  → ordinal enc  │
                    │                                │
                    └────────────┬───────────────────┘
                                 │
                         X_transformed
                    (32 numeric columns, all ready for ML)
```

### Why Pipeline prevents leakage (mechanically)

When you call `preprocessor.fit_transform(X_train)`:
- The median for `absences` is computed from 316 training rows
- The StandardScaler mean/std are computed from 316 training rows
- The OrdinalEncoder categories are learned from 316 training rows

When you later call `preprocessor.transform(X_test)`:
- The same 316-row median is used (test set doesn't influence it)
- The same 316-row mean/std is used
- Unknown categories map to -1 (not learned from test set)

The test set is **invisible** to all preprocessing statistics.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BUILD THE PREPROCESSING PIPELINE
# This is the exact pipeline used in train.py and the FastAPI backend
# ══════════════════════════════════════════════════════════════════

# ── Step 1: Numeric sub-pipeline ─────────────────────────────────
# SimpleImputer: fills NaN with the MEDIAN of the training column
# StandardScaler: z-score normalisation (mean=0, std=1)
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# ── Step 2: Categorical sub-pipeline ─────────────────────────────
# SimpleImputer: fills NaN with the MOST FREQUENT value (mode)
# OrdinalEncoder: maps each category string → integer
#   handle_unknown='use_encoded_value': unknown categories → -1 (production safety)
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

# ── Step 3: ColumnTransformer ─────────────────────────────────────
# Applies each sub-pipeline to its designated column subset
# remainder='drop': any column NOT listed is dropped from output
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,      NUMERIC_FEATURES),
        ('cat', categorical_pipeline,  CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

print('PIPELINE STRUCTURE')
print('=' * 50)
print('preprocessor')
print('└── ColumnTransformer')
print('    ├── num → numeric_pipeline')
print('    │   ├── SimpleImputer(strategy=median)')
print('    │   └── StandardScaler()')
print('    └── cat → categorical_pipeline')
print('        ├── SimpleImputer(strategy=most_frequent)')
print('        └── OrdinalEncoder(handle_unknown=use_encoded_value)')
print()
print(f'Input : {len(ALL_FEATURES)} raw columns')
print(f'Output: {len(ALL_FEATURES)} transformed numeric columns')
print('        (same count — OrdinalEncoder keeps 1 col per feature)')

In [ ]:
# ── FIT on training data ONLY, then TRANSFORM both sets ──────────
# This is the leakage-free pattern enforced by the Pipeline

# fit_transform: learns statistics from X_train, then transforms it
X_train_t = preprocessor.fit_transform(X_train)

# transform: applies the SAME statistics learned above to X_test
# (no new fitting — test set is fully isolated)
X_test_t = preprocessor.transform(X_test)

print('TRANSFORMATION COMPLETE')
print('=' * 50)
print(f'X_train_t shape: {X_train_t.shape}')
print(f'X_test_t  shape: {X_test_t.shape}')
print(f'dtype: {X_train_t.dtype}')
print()

# Verify no NaN values remain after imputation
nan_train = np.isnan(X_train_t).sum()
nan_test  = np.isnan(X_test_t).sum()
print(f'NaN in X_train_t: {nan_train}  {"✅" if nan_train == 0 else "⚠️ PROBLEM"}')
print(f'NaN in X_test_t : {nan_test}   {"✅" if nan_test == 0 else "⚠️ PROBLEM"}')

---
## Section 8 — Pipeline Validation

Before handing data to a model, always validate the output of your pipeline.
This is debugging discipline. You catch errors here, not after 30 minutes of training.

Things to check:
1. No NaN values remain
2. Shape is correct
3. Numeric features are centred near 0
4. Categorical features are small integers
5. Feature names are recoverable (for SHAP labelling)

In [ ]:
# ── Full validation suite ────────────────────────────────────────────────────

print('PIPELINE OUTPUT VALIDATION')
print('=' * 60)

# 1. Shape check
expected_cols = len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)
assert X_train_t.shape == (316, expected_cols), f'Shape mismatch: {X_train_t.shape}'
print(f'[✅] Shape correct: {X_train_t.shape}')

# 2. No NaN
assert not np.isnan(X_train_t).any(), 'NaN values found in transformed data!'
print(f'[✅] No NaN values in transformed training data')

# 3. Numeric columns are scaled
num_slice = X_train_t[:, :len(NUMERIC_FEATURES)]
mean_approx = np.abs(num_slice.mean(axis=0)).max()
std_approx  = num_slice.std(axis=0)
assert mean_approx < 0.1, f'Numeric columns not centred near 0! Max mean: {mean_approx:.3f}'
print(f'[✅] Numeric columns centred near 0 (max abs mean: {mean_approx:.4f})')

# 4. Categorical columns are small non-negative integers (+ -1 for unknowns)
cat_slice = X_train_t[:, len(NUMERIC_FEATURES):]
cat_max   = cat_slice.max()
assert cat_max < 10, f'Unexpected large categorical value: {cat_max}'
print(f'[✅] Categorical values in expected range [0, {cat_max:.0f}]')

# 5. Feature names recoverable
raw_names = preprocessor.get_feature_names_out()
clean_names = [n.split('__')[-1] for n in raw_names]
assert len(clean_names) == expected_cols
print(f'[✅] Feature names recoverable: {len(clean_names)} columns')

print()
print('All validation checks passed ✅')
print('Data is ready for model training.')

In [ ]:
# ── Inspect what the transformed matrix looks like ───────────────────────────
# Compare first row before and after transformation

print('BEFORE transformation (raw row 0):')
row_before = X_train.iloc[0]
print(row_before.to_string())
print()

print('AFTER transformation (row 0, first 10 features):')
transformed_row = X_train_t[0, :10]
for name, val in zip(clean_names[:10], transformed_row):
    print(f'  {name:<12}: {val:8.4f}')
print()

print('Numeric features → z-scores (negative = below mean, positive = above mean)')
print('Categorical features are encoded as integers in later columns.')

In [ ]:
# ── Visual check: distribution of all features post-transformation ────────────
df_transformed = pd.DataFrame(X_train_t[:, :len(NUMERIC_FEATURES)],
                               columns=NUMERIC_FEATURES)

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Post-Preprocessing Distributions — Numeric Features (z-scored)',
             fontsize=13, fontweight='bold')

for i, col in enumerate(NUMERIC_FEATURES):
    ax = axes[i]
    ax.hist(df_transformed[col], bins=15, color=ACCENT, edgecolor='#0f1117', alpha=0.85)
    ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_title(col, fontweight='bold', fontsize=9)
    ax.set_xlabel('z-score')
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('../plots/16_post_transform_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 All numeric features are now centred near 0.')
print('   Shapes are preserved — skewed features remain skewed after scaling.')
print('   (Scaling changes range, not shape. A log transform would change shape.)')

---
## Section 9 — Feature Engineering

### Should we create new features?

**Rule:** Only engineer a feature if you have a clear domain reason to believe
the combination is more predictive than the individual features.
Arbitrary feature engineering without domain justification is just noise.

### The one feature we engineer: `grade_delta = G2 - G1`

**Domain reasoning:**
- G1 (first period grade) and G2 (second period grade) are individually strong predictors
- But neither captures **trajectory** — is this student improving or declining?
- A student with G1=12, G2=7 (declining fast) is at very different risk than
  a student with G1=7, G2=12 (improving)
- `grade_delta = G2 - G1` captures this trend explicitly

```
Student A: G1=12, G2=7  → grade_delta = -5  (declining → high risk)
Student B: G1=7,  G2=12 → grade_delta = +5  (improving → lower risk)
Student C: G1=9,  G2=9  → grade_delta = 0   (stable → moderate risk)
```

**Why this is interview-friendly:**  
It shows you thought about the domain, not just the algorithm.
Interviewers love: *'I engineered one additional feature — the grade trajectory
(G2 - G1) — because the direction of change matters as much as the absolute level.'*

### What we deliberately did NOT engineer:
- `total_alcohol = Dalc + Walc` — marginal gain, XGBoost already combines features
- `absence_rate = absences / age` — no clear domain justification
- Polynomial features — would create hundreds of useless columns on a 395-row dataset

In [ ]:
# ── Create grade_delta feature ────────────────────────────────────────────────

X_fe = X.copy()  # feature-engineered copy
X_fe['grade_delta'] = X_fe['G2'] - X_fe['G1']

print('grade_delta = G2 - G1 (grade trajectory)')
print(f'Range: [{X_fe["grade_delta"].min():.0f}, {X_fe["grade_delta"].max():.0f}]')
print(f'Mean : {X_fe["grade_delta"].mean():.2f}')
print(f'Std  : {X_fe["grade_delta"].std():.2f}')
print()

# Compare grade_delta distribution for safe vs at-risk students
at_risk_delta  = X_fe.loc[y==1, 'grade_delta']
safe_delta     = X_fe.loc[y==0, 'grade_delta']

print(f'Mean grade_delta — At-Risk : {at_risk_delta.mean():.3f}')
print(f'Mean grade_delta — Safe    : {safe_delta.mean():.3f}')
print(f'Difference                 : {at_risk_delta.mean() - safe_delta.mean():.3f}')
print()
print('📌 At-risk students tend to have MORE NEGATIVE grade_delta')
print('   (declining grades) than safe students.')

In [ ]:
# ── Visualise grade_delta by risk class ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('grade_delta Feature Engineering Validation', fontweight='bold', fontsize=12)

# KDE comparison
ax = axes[0]
for risk_val, color, label in [(0, RISK_COLORS[0], 'Safe'), (1, RISK_COLORS[1], 'At Risk')]:
    data = X_fe.loc[y==risk_val, 'grade_delta']
    ax.hist(data, bins=15, density=True, alpha=0.35, color=color)
    data.plot.kde(ax=ax, color=color, linewidth=2.5, label=label)
ax.set_title('grade_delta Distribution by Risk Class', fontweight='bold')
ax.set_xlabel('grade_delta (G2 - G1)')
ax.set_ylabel('Density')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6, label='Δ=0')
ax.legend()

# Scatter: G1 vs G2 with delta colour coding
ax = axes[1]
scatter = ax.scatter(X_fe['G1'], X_fe['G2'],
                     c=X_fe['grade_delta'], cmap='RdYlGn',
                     alpha=0.6, s=30, vmin=-8, vmax=8)
ax.plot([0, 20], [0, 20], color='white', linestyle='--',
        linewidth=1, alpha=0.5, label='G1=G2 (no change)')
ax.set_xlabel('G1 (First Period Grade)')
ax.set_ylabel('G2 (Second Period Grade)')
ax.set_title('G1 vs G2 — Colour = grade_delta\n(Red=declining, Green=improving)',
             fontweight='bold')
plt.colorbar(scatter, ax=ax, label='grade_delta')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../plots/17_grade_delta_engineering.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Upper-right of scatter = strong students (G1 high AND G2 high → green).')
print('   Lower-left AND red = declining students — the highest-risk group.')

In [ ]:
# ── Build updated pipeline INCLUDING grade_delta ─────────────────────────────
# We add grade_delta to numeric features (it's a numeric delta)

NUMERIC_FEATURES_FE = NUMERIC_FEATURES + ['grade_delta']

X_fe_train, X_fe_test, _, _ = train_test_split(
    X_fe, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler',  StandardScaler())]),
         NUMERIC_FEATURES_FE),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('encoder', OrdinalEncoder(
                              handle_unknown='use_encoded_value', unknown_value=-1))]),
         CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

X_fe_train_t = preprocessor_fe.fit_transform(X_fe_train)
X_fe_test_t  = preprocessor_fe.transform(X_fe_test)

print(f'Pipeline with grade_delta → {X_fe_train_t.shape[1]} features')
print(f'(was {X_train_t.shape[1]} features before engineering)')
print()
print('NOTE: We will compare model performance WITH and WITHOUT grade_delta')
print('      in Phase 4 to decide whether to keep it in the final pipeline.')

---
## Section 10 — Save the Preprocessor

### Why save the preprocessor separately from the model?

In production:
1. **Training time:** `preprocessor.fit_transform(X_train)` → saves `preprocessor.pkl`
2. **Inference time:** FastAPI loads `preprocessor.pkl` → calls `preprocessor.transform(new_data)`

The preprocessing statistics (medians, stds, encoder categories) must be
**frozen at training time** and reused at inference time. This is exactly what
saving the fitted preprocessor achieves.

**Interview Q:** *'How do you ensure inference preprocessing matches training?'*  
**Answer:** *'I save the fitted sklearn ColumnTransformer as a .pkl artifact with joblib.
The FastAPI backend loads this artifact at startup. New CSV data is transformed
using the same statistics that were computed during training — ensuring perfect
consistency between training and serving.'*

### joblib vs pickle

Both serialize Python objects. `joblib` is preferred for sklearn objects because:
- More memory-efficient for large numpy arrays inside the preprocessor
- Slightly faster for loading large objects
- The sklearn team recommends it explicitly

In [ ]:
# ── Save the base preprocessor (without grade_delta) ─────────────────────────
# This is the version used in train.py and the FastAPI backend.
# (grade_delta will be evaluated in Phase 4 — if it improves Recall we add it)

PREPROCESSOR_PATH = '../models/preprocessor.pkl'

# Refit on FULL training set (not just 316 demo rows) before saving
# The preprocessor is already fitted above — we're saving that fitted state
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f'✅ Preprocessor saved to: {PREPROCESSOR_PATH}')

# ── Reload & validate the saved preprocessor ─────────────────────────────────
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

# Verify it transforms identically to the original
X_test_reloaded = loaded_preprocessor.transform(X_test)
assert np.allclose(X_test_t, X_test_reloaded), 'Loaded preprocessor produces different output!'
print('✅ Reload validation passed: saved and reloaded preprocessors produce identical output.')
print()

# ── Show what's stored inside the preprocessor ───────────────────────────────
print('WHAT IS STORED INSIDE preprocessor.pkl:')
print()

# Imputer statistics (medians)
num_imputer = loaded_preprocessor.named_transformers_['num'].named_steps['imputer']
print('Numeric imputer medians (first 5 features):')
for feat, median in zip(NUMERIC_FEATURES[:5], num_imputer.statistics_[:5]):
    print(f'  {feat:<12}: {median:.2f}')
print()

# Scaler statistics (mean and std)
scaler = loaded_preprocessor.named_transformers_['num'].named_steps['scaler']
print('Scaler mean and std (first 5 features):')
for feat, mean, std in zip(NUMERIC_FEATURES[:5], scaler.mean_[:5], scaler.scale_[:5]):
    print(f'  {feat:<12}: mean={mean:.3f}, std={std:.3f}')
print()

# Encoder categories
encoder = loaded_preprocessor.named_transformers_['cat'].named_steps['encoder']
print('Ordinal encoder categories (first 3 features):')
for feat, cats in zip(CATEGORICAL_FEATURES[:3], encoder.categories_[:3]):
    print(f'  {feat:<12}: {list(cats)}')

In [ ]:
# ── Demonstrate inference simulation ─────────────────────────────────────────
# Simulate what happens when FastAPI receives a CSV row from faculty
# and needs to preprocess it before sending to the XGBoost model.

print('INFERENCE SIMULATION — one new student row')
print('=' * 55)

# Simulated new student data as a dict (what FastAPI would receive from CSV upload)
new_student_raw = {
    'age': 17, 'Medu': 2, 'Fedu': 1, 'traveltime': 2,
    'studytime': 1,  # Very low study time — risk flag
    'failures': 2,   # 2 past failures — major risk flag
    'famrel': 3, 'freetime': 4, 'goout': 4,
    'Dalc': 2, 'Walc': 3, 'health': 2,
    'absences': 18,  # High absences — risk flag
    'G1': 7, 'G2': 5,  # Low and declining grades
    'school': 'GP', 'sex': 'M', 'address': 'U', 'famsize': 'GT3',
    'Pstatus': 'T', 'Mjob': 'other', 'Fjob': 'other',
    'reason': 'course', 'guardian': 'mother',
    'schoolsup': 'yes', 'famsup': 'no', 'paid': 'no',
    'activities': 'no', 'nursery': 'yes', 'higher': 'no',
    'internet': 'yes', 'romantic': 'no'
}

# Convert to DataFrame (same format as pandas would parse from a CSV row)
new_student_df = pd.DataFrame([new_student_raw])
print('Raw input from faculty CSV:')
print(f'  studytime={new_student_raw["studytime"]} (very low), failures={new_student_raw["failures"]}, absences={new_student_raw["absences"]}')
print(f'  G1={new_student_raw["G1"]}, G2={new_student_raw["G2"]} (declining)')
print()

# Apply the saved preprocessor (exact same call the FastAPI backend makes)
new_student_t = loaded_preprocessor.transform(new_student_df[ALL_FEATURES])

print(f'After preprocessing: shape={new_student_t.shape}')
print(f'All values are numeric: {np.isnan(new_student_t).sum() == 0}')
print()
print('First 5 transformed values:')
feature_names_clean = [n.split('__')[-1] for n in loaded_preprocessor.get_feature_names_out()]
for name, val in zip(feature_names_clean[:5], new_student_t[0, :5]):
    print(f'  {name:<12}: {val:8.4f}')
print()
print('✅ Inference pipeline works correctly.')
print('   This student has multiple risk signals — model will likely score them HIGH RISK.')

---
## Section 11 — Interview Questions from Preprocessing

Every question below has been asked at Meesho, AmEx, Navi, and similar product companies.
These come directly from what we built in this notebook.

In [ ]:
interview_qa = [
    {
        'Q': 'Q1. What is data leakage and how did you prevent it?',
        'A': """Data leakage occurs when information from the test set (or future data)
leaks into the training process, making model performance appear better than it truly is.

Two types I guard against in FAILSAFE:

1. TARGET LEAKAGE: G3 (final grade) is the source of our label at_risk.
   If I included G3 as a feature, the model would learn a trivial rule
   (G3 < 10 → at_risk=1) and achieve ~100% accuracy on test data — useless.
   Prevention: explicitly drop G3 before creating X.

2. PREPROCESSING LEAKAGE: If I scaled all data BEFORE splitting, the scaler
   would learn the mean/std from ALL 395 rows including the test set.
   The test set would have indirectly influenced the preprocessing.
   Prevention: use sklearn Pipeline — it enforces fit on train, transform on test.
   preprocessor.fit_transform(X_train) → preprocessor.transform(X_test)"""
    },
    {
        'Q': 'Q2. Why did you use a Pipeline instead of applying transformations manually?',
        'A': """Three reasons:

1. LEAKAGE PREVENTION: Pipeline enforces fit-only-on-train semantics automatically.
   Manual transforms require discipline; Pipeline makes it structurally impossible
   to fit on test data by accident.

2. PRODUCTION CONSISTENCY: I save the fitted Pipeline as preprocessor.pkl.
   When FastAPI receives a new CSV, it calls preprocessor.transform(new_data).
   The EXACT same median, std, and encoder categories used during training are
   applied to inference data — guaranteed consistency.

3. MAINTAINABILITY: One object represents the entire preprocessing logic.
   To add a new step, I add one line to the Pipeline — no risk of forgetting
   to apply it in both training and inference code."""
    },
    {
        'Q': 'Q3. Why OrdinalEncoder and not OneHotEncoder?',
        'A': """My primary model is XGBoost, which is tree-based.

Tree models make decisions by finding the best SPLIT THRESHOLD at each node:
  e.g., 'if Mjob_encoded < 2, go left; else go right'

Whether 'teacher' maps to 0 or 4 doesn't matter — the tree finds the optimal
split regardless. OrdinalEncoder keeps one column per feature = 17 categorical
columns stay as 17 columns.

OneHotEncoder would expand 17 categorical features into ~45+ binary columns
for no benefit to XGBoost, and would make SHAP interpretation harder.

IMPORTANT CAVEAT: For Logistic Regression (our baseline), OrdinalEncoder IS wrong
because LR is a linear model that assumes numerical ordering. In a production system
serving only LR, I would use OneHotEncoder. Here, since XGBoost is the production
model, OrdinalEncoder is the right choice."""
    },
    {
        'Q': 'Q4. Does XGBoost need feature scaling? Why did you include it?',
        'A': """Strictly: NO, XGBoost does not need scaling. Tree-based algorithms
are invariant to monotonic transformations of features — the split threshold
adjusts automatically.

I included scaling for three practical reasons:
1. The same preprocessor.pkl serves both XGBoost AND Logistic Regression (baseline).
   LR absolutely needs scaling for gradient descent to converge properly.
2. Scaled SHAP values are slightly more comparable in magnitude across features.
3. Future-proofing: if we add an SVM or neural network baseline, scaling is already handled.

The cost is zero — scaling a numeric array is O(n) and adds negligible latency.
The benefit is a more robust, generalizable pipeline."""
    },
    {
        'Q': 'Q5. What is stratified split and when is it necessary?',
        'A': """A stratified split ensures the class ratio in the training set matches
the class ratio in the test set (and the original dataset).

Without stratification on imbalanced data:
  Random chance could put 90% of at-risk students in the training set
  → test set has only 10% at-risk students
  → test Recall score is artificially high (few positives to miss)
  → model is NOT actually good at detecting at-risk students

With stratify=y:
  Both train (~33% at-risk) and test (~33% at-risk) reflect the real distribution
  → Recall, Precision, F1 on test set are realistic estimates of production performance

It's mandatory whenever minority class < 40% of data — which is our case (~33%)."""
    },
    {
        'Q': 'Q6. Why did you engineer grade_delta = G2 - G1?',
        'A': """G1 and G2 individually measure LEVEL — how high the grade is.
grade_delta measures TRAJECTORY — is the student improving or declining?

Two students with identical G2=8 are very different:
  Student A: G1=12, G2=8 (grade_delta=-4) — declining sharply → HIGH risk
  Student B: G1=4,  G2=8 (grade_delta=+4) — recovering strongly → LOWER risk

XGBoost can in principle discover this interaction between G1 and G2 on its own,
but explicitly engineering it:
1. Reduces the search space the model needs to explore
2. Makes the feature immediately interpretable in SHAP: 'Declining grades
   increased this student's risk score by 0.23'
3. Demonstrates domain knowledge in the interview"""
    },
    {
        'Q': 'Q7. How does your preprocessing handle new/unseen data in production?',
        'A': """Three mechanisms protect against production data drift:

1. MISSING VALUES: SimpleImputer fills NaN with training medians/modes.
   If faculty CSV has blanks, we fill them rather than crashing.

2. UNKNOWN CATEGORIES: OrdinalEncoder with handle_unknown='use_encoded_value'
   maps new category values (e.g., a new school name) to -1.
   XGBoost will still make a prediction — it just won't use this feature's
   signal. Better than crashing.

3. COLUMN VALIDATION: In the FastAPI upload endpoint, I validate that all
   required columns are present BEFORE passing to the preprocessor.
   Missing required columns → 400 Bad Request with a clear error message.

This three-layer defense is why I separate validation (FastAPI) from
transformation (preprocessor.pkl) — each layer has one clear responsibility."""
    },
    {
        'Q': 'Q8. What is ColumnTransformer and why is it better than separate pipelines?',
        'A': """ColumnTransformer applies DIFFERENT transformations to different SUBSETS
of columns simultaneously, then CONCATENATES the results into one matrix.

Without ColumnTransformer:
  You'd have to split X into numeric and categorical DataFrames,
  transform each separately, then manually concatenate — error-prone and messy.

With ColumnTransformer:
  One object handles the entire heterogeneous feature matrix.
  It fits and transforms in one call.
  Column order is deterministic and stored inside the object.
  get_feature_names_out() gives you the final column names for SHAP labelling.

It's the standard sklearn pattern for any dataset with mixed feature types —
which is virtually every real-world dataset."""
    },
]

for qa in interview_qa:
    print('━' * 70)
    print(f"\n🎯 {qa['Q']}\n")
    print(f"💡 ANSWER:\n{qa['A']}\n")

---
## ✅ Phase 3 Complete — Preprocessing Summary

| Decision | Choice | Why |
|----------|--------|-----|
| **Feature selection** | 15 numeric + 17 categorical | Domain-driven; G3 dropped (leakage) |
| **Split ratio** | 80/20 | Standard; 316 train samples is sufficient |
| **Stratification** | `stratify=y` | Class imbalance ~33%; mandatory |
| **Numeric imputation** | Median | `absences` is right-skewed; robust to outliers |
| **Categorical imputation** | Mode | Most common category = safest default |
| **Categorical encoding** | OrdinalEncoder | XGBoost is tree-based; ordering irrelevant |
| **Scaling** | StandardScaler | Needed for LR baseline; zero cost for XGBoost |
| **Feature engineering** | `grade_delta = G2-G1` | Captures trajectory, not just level |
| **Pipeline serialisation** | `joblib.dump` | Frozen training statistics for inference |

---

**Files produced by this notebook:**
- `models/preprocessor.pkl` — fitted ColumnTransformer, ready for FastAPI
- `plots/14_train_test_split.png` — class ratio validation
- `plots/15_scaling_demo.png` — before/after StandardScaler
- `plots/16_post_transform_distributions.png` — all features post-transform
- `plots/17_grade_delta_engineering.png` — grade_delta validation

---

**Next: Phase 4 — `03_modeling.ipynb`**  
We train all three models (LR, RF, XGBoost), evaluate with Accuracy/Precision/Recall/F1/ROC-AUC,
explain the confusion matrix, discuss threshold tuning, and justify our final model selection.